# [16.4] TokenSHAP and TokenShapley

## Core question

If a prompt score depends on both a target token and a context token, can exact token-position Shapley values recover the fair split before we trust sampled TokenSHAP or a trained model?

## Learning objectives

By the end, you should be able to:

1. Enumerate complete token-position coalitions.
2. Build masked-token value tables with a stable baseline.
3. Compute exact Shapley values and check efficiency.
4. Estimate TokenSHAP with sampled random orderings.
5. Interpret a CUDA trained-token-scorer report with a shuffled-label negative control.

> Difficulty: 4/5  
> Importance: 4/5

<img src="../../instructions/assets/tokenshap_validation_loop.svg" width="760">

This notebook uses the prompt `The capital is Paris`. The toy score gives one point for `Paris` and two interaction points when `capital` and `Paris` are both present. The exact token Shapley result should be `[0.0, 1.0, 0.0, 2.0]`.

<details><summary>Help - why this toy game?</summary>

It is small enough to enumerate exactly, but it still contains the failure mode that makes token attribution interesting: some credit belongs to a context token because it makes the target token valuable.

</details>


## Setup

Run this once before the exercises. The visible tests are small and deterministic.

<details><summary>Expected output</summary>

No printed output. Imports should succeed and the report dataclasses should be defined.

</details>


In [1]:
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
import itertools
import json
import math
import random
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part4_tokenshap_token_shapley"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_tokenshap_token_shapley.tests as tests

Coalition = frozenset[int]
MAIN = True
TOKENS = ("The", "capital", "is", "Paris")
MASK_TOKEN = "[MASK]"


@dataclass(frozen=True)
class TokenShapleySamplingReport:
    tokens: tuple[str, ...]
    exact_values: t.Tensor
    sampled_values: t.Tensor
    max_abs_error: float
    top_token: str
    sampled_top_token: str
    rank_matches: bool
    approximates_exact: bool


@dataclass(frozen=True)
class TokenBaselineReport:
    full_score: float
    baseline_score: float
    total_delta: float
    shapley_sum: float
    efficiency_error: float
    satisfies_efficiency: bool


## Exercise 1 - enumerate token-position coalitions

TokenSHAP starts from every subset of prompt positions. Keep the empty coalition: it is the fully masked baseline.

<details><summary>Help - what invariant should this prove?</summary>

For four tokens there must be exactly `2**4 = 16` coalitions, with no duplicates. If the empty coalition is missing, the baseline is no longer explicit.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_all_coalitions_enumerates_complete_powerset` passed!
```

</details>

<details><summary>Common bugs</summary>

- Returning only nonempty coalitions.
- Accepting zero players.
- Returning mutable sets rather than `frozenset` keys.

</details>

<details><summary>Solution</summary>

Loop over subset sizes and use `itertools.combinations` to produce `frozenset` coalitions.

</details>


In [2]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    """Return every coalition for `num_players`, ordered by size."""
    if num_players <= 0:
        raise ValueError("num_players must be positive.")
    players = range(num_players)
    coalitions: list[Coalition] = []
    for size in range(num_players + 1):
        coalitions.extend(frozenset(group) for group in itertools.combinations(players, size))
    return tuple(coalitions)


if MAIN:
    tests.test_all_coalitions_enumerates_complete_powerset(all_coalitions)


All tests in `test_all_coalitions_enumerates_complete_powerset` passed!


## Exercise 2 - implement the context-target score

The toy score gives one point for `Paris` and two extra points when `capital` and `Paris` are both present.

<details><summary>Help - why does this create an attribution test?</summary>

A context token should receive credit only because it changes the value of the target token. This makes the expected fair split exact: `capital` gets one point and `Paris` gets two.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_keyword_interaction_token_score_is_target_context_game` passed!
```

</details>

<details><summary>Common bugs</summary>

- Giving `capital` credit when `Paris` is absent.
- Adding the interaction twice.
- Checking original positions instead of the masked token tuple.

</details>

<details><summary>Solution</summary>

Check token presence, add the target point if `Paris` appears, then add the interaction only when both tokens appear.

</details>


In [3]:
def keyword_interaction_token_score(
    tokens: Sequence[str],
    *,
    target_token: str = "Paris",
    context_token: str = "capital",
    target_weight: float = 1.0,
    interaction_weight: float = 2.0,
) -> float:
    """Toy prompt score with one target token and one context-target interaction."""
    token_set = set(tokens)
    target_present = target_token in token_set
    context_present = context_token in token_set
    score = target_weight if target_present else 0.0
    if target_present and context_present:
        score += interaction_weight
    return score


if MAIN:
    tests.test_keyword_interaction_token_score_is_target_context_game(
        keyword_interaction_token_score,
    )


All tests in `test_keyword_interaction_token_score_is_target_context_game` passed!


## Exercise 3 - build masked token coalition values

For each coalition, preserve selected positions and replace absent positions with `[MASK]` before scoring.

<details><summary>Help - why mask rather than delete?</summary>

Deleting tokens changes positions. This section attributes prompt positions, so the baseline should preserve sequence length.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_token_coalition_values_masks_absent_positions` passed!
```

</details>

<details><summary>What you should see</summary>

The empty coalition scores `0.0`, the `Paris`-only coalition scores `1.0`, and the `capital` plus `Paris` coalition scores `3.0`.

</details>

<details><summary>Common bugs</summary>

- Using token strings as players.
- Dropping masked positions.
- Forgetting to validate empty token sequences.

</details>

<details><summary>Solution</summary>

Enumerate coalitions, construct the masked tuple position-by-position, and score each tuple.

</details>


In [4]:
def token_coalition_values(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
) -> dict[Coalition, float]:
    """Build a complete masked-position coalition table for token attribution."""
    token_tuple = tuple(tokens)
    if not token_tuple:
        raise ValueError("tokens must be nonempty.")
    values: dict[Coalition, float] = {}
    for coalition in all_coalitions(len(token_tuple)):
        masked_tokens = tuple(
            token if index in coalition else mask_token
            for index, token in enumerate(token_tuple)
        )
        values[coalition] = float(score_fn(masked_tokens))
    return values


if MAIN:
    tests.test_token_coalition_values_masks_absent_positions(token_coalition_values)


All tests in `test_token_coalition_values_masks_absent_positions` passed!


## Exercise 4 - compute exact token Shapley values

Average each token's marginal contribution over all coalitions that do not already contain that token.

<details><summary>Help - why does `capital` get nonzero credit?</summary>

Across orderings, the interaction is sometimes created when `capital` arrives and sometimes when `Paris` arrives. Exact Shapley splits that two-point interaction evenly.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_exact_token_shapley_values_splits_context_target_interaction` passed!
```

</details>

<details><summary>What you should see</summary>

```text
The      0.0
capital  1.0
is       0.0
Paris    2.0
```

</details>

<details><summary>Common bugs</summary>

- Forgetting the factorial Shapley weight.
- Including coalitions that already contain the player.
- Implementing leave-one-out rather than Shapley values.

</details>

<details><summary>Solution</summary>

Normalize the complete coalition table, then loop over players, subset sizes, and background coalitions with the Shapley factorial weight.

</details>


In [5]:
def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    """Normalize coalition keys and require a complete value table."""
    values = {frozenset(key): float(value) for key, value in coalition_values.items()}
    expected = set(all_coalitions(num_players))
    missing = expected - set(values)
    if missing:
        raise ValueError(f"coalition value table is missing {len(missing)} coalitions.")
    return values


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    """Compute exact Shapley values from a complete coalition-value table."""
    values = normalize_coalition_values(coalition_values, num_players=num_players)
    shapley = t.zeros(num_players, dtype=t.float64)
    denominator = math.factorial(num_players)
    for player in range(num_players):
        other_players = [index for index in range(num_players) if index != player]
        for coalition_size in range(num_players):
            weight = (
                math.factorial(coalition_size)
                * math.factorial(num_players - coalition_size - 1)
                / denominator
            )
            for group in itertools.combinations(other_players, coalition_size):
                coalition = frozenset(group)
                shapley[player] += weight * (
                    values[coalition | {player}] - values[coalition]
                )
    return shapley


def exact_token_shapley_values(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
) -> t.Tensor:
    """Compute exact Shapley values over token positions."""
    token_tuple = tuple(tokens)
    values = token_coalition_values(token_tuple, score_fn, mask_token=mask_token)
    return exact_shapley_values(values, num_players=len(token_tuple))


if MAIN:
    tests.test_exact_token_shapley_values_splits_context_target_interaction(
        exact_token_shapley_values,
    )


All tests in `test_exact_token_shapley_values_splits_context_target_interaction` passed!


## Exercise 5 - check masked-baseline efficiency

The sum of Shapley values should equal `score(full prompt) - score(masked prompt)`.

<details><summary>Help - why keep the numeric report?</summary>

The boolean is not enough for debugging. The full score, baseline score, total delta, Shapley sum, and efficiency error tell you which part of the pipeline broke.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_token_baseline_report_checks_efficiency` passed!
```

</details>

<details><summary>Common bugs</summary>

- Comparing the Shapley sum to the full score instead of full-minus-baseline.
- Using a different mask token from the exact-value call.
- Returning only a boolean.

</details>

<details><summary>Solution</summary>

Compute exact values, full score, masked score, and check the absolute efficiency error against the tolerance.

</details>


In [6]:
def token_baseline_report(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
    tolerance: float = 1e-9,
) -> TokenBaselineReport:
    """Check token Shapley efficiency against masked and full prompt scores."""
    token_tuple = tuple(tokens)
    baseline_tokens = tuple(mask_token for _ in token_tuple)
    exact = exact_token_shapley_values(token_tuple, score_fn, mask_token=mask_token)
    full_score = float(score_fn(token_tuple))
    baseline_score = float(score_fn(baseline_tokens))
    total_delta = full_score - baseline_score
    shapley_sum = float(exact.sum().item())
    efficiency_error = abs(shapley_sum - total_delta)
    return TokenBaselineReport(
        full_score=full_score,
        baseline_score=baseline_score,
        total_delta=total_delta,
        shapley_sum=shapley_sum,
        efficiency_error=efficiency_error,
        satisfies_efficiency=efficiency_error <= tolerance,
    )


if MAIN:
    tests.test_token_baseline_report_checks_efficiency(token_baseline_report)


All tests in `test_token_baseline_report_checks_efficiency` passed!


## Exercise 6 - estimate TokenSHAP with sampled orderings

Average marginal contributions over random player orderings, then compare the estimate to the exact values.

<details><summary>Help - why is exact parity still required?</summary>

A top-token match can hide numeric drift. On small games, sampled TokenSHAP should be checked against exact values before you use the same estimator on prompts too long to enumerate.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_token_shapley_sampling_report_matches_exact_ranking` passed!
```

</details>

<details><summary>What you should see</summary>

```text
exact   = [0.0, 1.0, 0.0, 2.0]
sampled = [0.0, 1.05078125, 0.0, 1.94921875]
top token = Paris
```

</details>

<details><summary>Common bugs</summary>

- Sampling coalitions uniformly instead of token orderings.
- Forgetting to update the coalition after adding a player.
- Ignoring numeric error once the top token matches.

</details>

<details><summary>Solution</summary>

Use a seeded `random.Random`, sample a full token ordering each time, accumulate each player's marginal contribution, and divide by the number of samples.

</details>


In [7]:
def sampled_permutation_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    num_samples: int,
    seed: int = 0,
) -> t.Tensor:
    """Estimate Shapley values by averaging random player orderings."""
    if num_samples <= 0:
        raise ValueError("num_samples must be positive.")
    values = normalize_coalition_values(coalition_values, num_players=num_players)
    rng = random.Random(seed)
    players = tuple(range(num_players))
    totals = t.zeros(num_players, dtype=t.float64)
    for _ in range(num_samples):
        coalition: Coalition = frozenset()
        for player in rng.sample(players, k=num_players):
            with_player = coalition | {player}
            totals[player] += values[with_player] - values[coalition]
            coalition = with_player
    return totals / num_samples


def sampled_token_shapley_values(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
    num_samples: int = 256,
    seed: int = 0,
) -> t.Tensor:
    """Estimate token-position Shapley values with sampled permutations."""
    token_tuple = tuple(tokens)
    values = token_coalition_values(token_tuple, score_fn, mask_token=mask_token)
    return sampled_permutation_shapley_values(
        values,
        num_players=len(token_tuple),
        num_samples=num_samples,
        seed=seed,
    )


def token_shapley_sampling_report(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
    num_samples: int = 256,
    seed: int = 0,
    tolerance: float = 0.15,
) -> TokenShapleySamplingReport:
    """Compare sampled TokenSHAP values with exact token Shapley values."""
    token_tuple = tuple(tokens)
    exact = exact_token_shapley_values(token_tuple, score_fn, mask_token=mask_token)
    sampled = sampled_token_shapley_values(
        token_tuple,
        score_fn,
        mask_token=mask_token,
        num_samples=num_samples,
        seed=seed,
    )
    max_abs_error = float((exact - sampled).abs().max().item())
    top_index = int(exact.argmax().item())
    sampled_top_index = int(sampled.argmax().item())
    return TokenShapleySamplingReport(
        tokens=token_tuple,
        exact_values=exact,
        sampled_values=sampled,
        max_abs_error=max_abs_error,
        top_token=token_tuple[top_index],
        sampled_top_token=token_tuple[sampled_top_index],
        rank_matches=top_index == sampled_top_index,
        approximates_exact=max_abs_error <= tolerance,
    )


if MAIN:
    tests.test_token_shapley_sampling_report_matches_exact_ranking(
        token_shapley_sampling_report,
    )


All tests in `test_token_shapley_sampling_report_matches_exact_ranking` passed!


## Exercise 7 - bundle a notebook contract

The final smoke report should expose the coalition table landmarks, exact values, efficiency report, and sampled approximation.

<details><summary>Help - why return JSON-like dictionaries?</summary>

The committed verification report is JSON. Returning plain numbers and lists makes the notebook result easy to compare with the report and prevents hidden tensor serialization issues.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Common bugs</summary>

- Returning tensors directly from the smoke contract.
- Omitting the efficiency report.
- Running sampled TokenSHAP with a different seed from the expected output.

</details>

<details><summary>Solution</summary>

Return `coalitions`, `exact`, and `sampled` dictionaries built from the helper functions above.

</details>


In [8]:
def _tensor_report(report) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def token_coalition_smoke_test() -> dict:
    values = token_coalition_values(TOKENS, keyword_interaction_token_score)
    return {
        "empty": values[frozenset()],
        "target_only": values[frozenset({3})],
        "context_and_target": values[frozenset({1, 3})],
    }


def exact_token_shapley_smoke_test() -> dict:
    exact = exact_token_shapley_values(TOKENS, keyword_interaction_token_score)
    return {
        "tokens": list(TOKENS),
        "exact_values": exact.tolist(),
        "baseline": token_baseline_report(
            TOKENS,
            keyword_interaction_token_score,
        ).__dict__,
    }


def sampled_tokenshap_smoke_test() -> dict:
    report = token_shapley_sampling_report(
        TOKENS,
        keyword_interaction_token_score,
        num_samples=512,
        seed=0,
        tolerance=0.1,
    )
    return _tensor_report(report)


def run_smoke_test(cpu: bool = True) -> dict:
    """Return the visible notebook contract for this section."""
    _ = cpu
    return {
        "coalitions": token_coalition_smoke_test(),
        "exact": exact_token_shapley_smoke_test(),
        "sampled": sampled_tokenshap_smoke_test(),
    }


if MAIN:
    tests.test_notebook_contract(run_smoke_test)


All tests in `test_notebook_contract` passed!


## Committed CUDA report

Now inspect the committed CUDA report. This is not a substitute for the exercises above; it checks that the same exact attribution logic recovers the toy ground truth when coalition values come from a trained CUDA token scorer.

<details><summary>Expected output</summary>

```text
preflight_passed: true
true_exact_values: [0.0, 1.0, 0.0, 2.0]
model_exact_values: approximately [0.0, 1.0, 0.0, 2.0]
sampled_top_token: Paris
shuffled_control_rejected: true
```

</details>

<details><summary>Help - what does this prove?</summary>

It proves a finite trained-token-scorer model organism recovers the exact token Shapley result from real forward passes. It does not prove broad LLM prompt attribution.

</details>

<details><summary>Solution</summary>

Read `verification_report.json`, assert the accepted CUDA metrics, and return the `metrics.gpu_test` dictionary.

</details>


In [9]:
def load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    assert report["accepted"] is True and report["tests_passed"] is True
    assert gpu["cuda_available"] is True and gpu["preflight_passed"] is True
    assert gpu["true_exact_values"] == [0.0, 1.0, 0.0, 2.0]
    assert gpu["top_token"] == "Paris"
    assert gpu["sampled_top_token"] == "Paris"
    assert gpu["shuffled_control_rejected"] is True
    tests.test_committed_gpu_report_matches_token_shapley_contract(gpu)
    return report


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    _ = max_vram_gb
    return load_committed_gpu_report()["metrics"]["gpu_test"]


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


if MAIN:
    gpu_report = run_gpu_test()
    {
        "fit_mse": gpu_report["fit_mse"],
        "true_exact_values": gpu_report["true_exact_values"],
        "model_exact_values": gpu_report["model_exact_values"],
        "sampled_values": gpu_report["sampled_values"],
        "sampled_top_token": gpu_report["sampled_top_token"],
        "shuffled_control_error": gpu_report["shuffled_control_error"],
        "shuffled_control_rejected": gpu_report["shuffled_control_rejected"],
        "peak_vram_gb": gpu_report["peak_vram_gb"],
    }


All tests in `test_committed_gpu_report_matches_token_shapley_contract` passed!


## Signature Result

<img src="../../instructions/assets/tokenshap_signature_result.svg" width="760">

The exact result is `[0.0, 1.0, 0.0, 2.0]`: filler tokens get zero, `capital` gets half of the context-target interaction, and `Paris` gets the target point plus the other half.

<details><summary>Interpreting the result</summary>

The table is meaningful because it has controls: exact enumeration,
efficiency error `0.0`, sampled parity, and shuffled-label rejection. If the
shuffled-label scorer had passed, this would be a negative result rather than
a positive TokenSHAP demonstration.

</details>

## Limitations

This does not prove TokenSHAP is reliable on arbitrary LLM prompts, that `[MASK]` is always the right baseline, or that input-token attribution is a causal circuit explanation.

## Bonus / Exploring Anomalies

Try a repeated token, a different mask token, or a longer prompt. Plot sampled error against sample count before making any real-model claim.
